In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import json
import pandas as pd
from sqlalchemy import create_engine, text

configs = json.loads((Path().resolve().parent.joinpath("config.json").read_text()))

In [ ]:
engine = create_engine('mssql+pyodbc://@' + configs["sql_server"] + '/' + configs["sql_db"] + '?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server')
conn = engine.connect()

query = text("""SELECT
      dim_guid.username
   ,  MAX(fact_value.timestamp) AS 'timestamp'
   ,  SUM(fact_value.value) AS 'value'
FROM dbo.fact_value
   JOIN dbo.dim_guid ON fact_value.dim_guid_id = dim_guid.id
   JOIN dbo.dim_stat ON fact_value.dim_stat_id = dim_stat.id
   JOIN dbo.dim_type ON fact_value.dim_type_id = dim_type.id
WHERE
         dim_type.type = 'custom'
   AND   dim_stat.stat = 'play_time'
GROUP BY
      dim_guid.username
ORDER BY 3 DESC""")

df = pd.DataFrame(conn.execute(query))
df

In [ ]:
# adding percent, cumulative percent, and grouping players after 90% of cumulative play time
df["pct"] = df["value"] / df["value"].sum()
df["pct_cum"] = df["pct"].cumsum()
df["group"] = df.apply(lambda x: "Other" if x["pct_cum"] >= 0.9 else x["username"], axis=1)
df["player_count"] = 1
df

In [ ]:
def clean_label(row) -> str:
  return (f"""
    {row["group"]}{f" ({int(row["player_count"])})" if row["group"] == "Other" else ""}
    {row["time"]} ({round(100 * row["pct"], 2)}%)
    {row["timestamp"]}
    """)

def clean_time(row) -> str:
  total_seconds = row["value"] / 20
  hours = 0 if total_seconds < 3600 else int(total_seconds / 3600)
  minutes = int((total_seconds % 3600) / 60)
  seconds = int(total_seconds % 60)
  return f"{hours:,}:{str(minutes).rjust(2, "0")}:{str(seconds).rjust(2, "0")}"

In [ ]:
view = pd.concat((
   df[df["group"] != "Other"]
   , df[df["group"] == "Other"]
      .groupby("group", as_index=False)
      .agg({"timestamp": "max", "value": "sum", "pct": "sum", "player_count": "sum"})
   ))
view["time"] = view.apply(clean_time, axis=1)
view["label"] = view.apply(clean_label, axis=1)
view

In [ ]:
plt.pie(view["value"], labels=view["label"].astype(str).tolist(), counterclock=False, startangle=90, radius=3)